In [1]:
import json
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [2]:
# install dash
!pip install dash==2.8.1
import dash
from dash import Dash, dcc, html, Input, Output
import plotly.express as px

/opt/anaconda3/lib/python3.13/site-packages/dash/dash.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, parse_version


In [3]:
from pymongo import MongoClient
client = MongoClient('localhost', 27017)

db = client['weather_analysis']
weather_col = db['weather']

In [4]:
# Drop collection to avoid duplicates on re-run
weather_col.drop()

# Load and insert the JSON file
with open('weather-1.json', 'r') as f:
    weather_data = json.load(f)

result = weather_col.insert_many(weather_data)
print(f'Inserted {len(result.inserted_ids)} documents into weather_analysis.weather')

# Show a sample document
sample = weather_col.find_one({}, {'_id': 0})
print(sample)

Inserted 1825 documents into weather_analysis.weather
{'Date': '2024-01-01T00:00:00.000', 'City': 'Seattle', 'Temperature_F': 51.5, 'Humidity_%': 91.6, 'WindSpeed_mph': 9.3, 'Precipitation_in': 2.23}


In [5]:
url = 'https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population'
response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
response.raise_for_status()

soup = BeautifulSoup(response.text, 'html.parser')

# Find the correct table containing city data
tables = soup.find_all('table')
table = None
for t in tables:
    text = t.get_text()
    if 'New York' in text and 'Los Angeles' in text and 'Chicago' in text:
        table = t
        break

# Parse using colspan method (professor's style)
row_headers = []
row_data = []
for row in table.find_all('tr'):
    for th in row.find_all('th'):
        row_headers.append(th.text.strip())
    data = []
    for td in row.find_all('td'):
        colspan_val = int(td.get('colspan', 1))
        if colspan_val == 1:
            data.append(td.text.strip())
        elif colspan_val > 1:
            for i in range(colspan_val):
                data.append(td.text.strip())
    row_data.append(data)

row_data = row_data[1:]  # skip header row

print('Sample row:')
for i, val in enumerate(row_data[1]):
    print(f'  col[{i}] = {val}')

Sample row:
  col[0] = New York[c]
  col[1] = NY
  col[2] = 8,584,629
  col[3] = 8,804,190
  col[4] = −2.49%
  col[5] = 300.5
  col[6] = 778.3
  col[7] = 29,298
  col[8] = 11,312
  col[9] = 40°40′N 73°56′W﻿ / ﻿40.66°N 73.94°W﻿ / 40.66; -73.94﻿ (New York City)


In [6]:
TARGET_CITIES = {'Seattle', 'Chicago', 'Denver', 'Phoenix', 'Miami'}

city_list    = []
st_list      = []
pop2025_list = []
pop2020_list = []
land_mi_list = []
land_km_list = []

for row in row_data:
    if len(row) < 5:
        continue
    city_name = re.sub(r'\[.*?\]', '', row[0]).strip()
    if city_name not in TARGET_CITIES:
        continue
    city_list.append(city_name)
    st_list.append(row[1])
    pop2025_list.append(row[2])
    pop2020_list.append(row[3])
    # row[4] = Change      DROPPED
    land_mi_list.append(row[5] if len(row) > 5 else '')
    land_km_list.append(row[6] if len(row) > 6 else '')
    # row[7], row[8] = density DROPPED
    # row[9] = location  DROPPED

city_data = {
    'City':                     city_list,
    'ST':                       st_list,
    'Population_2025_estimate': pop2025_list,
    'Population_2020_census':   pop2020_list,
    'Land_area_mi2_2020':       land_mi_list,
    'Land_area_km2_2020':       land_km_list
}
city_df = pd.DataFrame(city_data)
print(city_df)

cities_col = db['cities']
cities_col.drop()
city_docs = city_df.to_dict('records')
result2 = cities_col.insert_many(city_docs)
print(f'Inserted {len(result2.inserted_ids)} documents into weather_analysis.cities')

      City  ST Population_2025_estimate Population_2020_census  \
0  Chicago  IL                2,731,585              2,746,388   
1  Phoenix  AZ                1,665,481              1,608,139   
2  Seattle  WA                  784,777                737,015   
3   Denver  CO                  740,613                715,522   
4    Miami  FL                  489,812                442,241   

  Land_area_mi2_2020 Land_area_km2_2020  
0              227.7              589.7  
1              518.0            1,341.6  
2               83.8              217.0  
3              153.1              396.5  
4               36.0               93.2  
Inserted 5 documents into weather_analysis.cities


In [7]:
# Q1 Average temperature & humidity per city for January 2025
pipeline_q1 = [
    {
        '$addFields': {
            'DateParsed': {'$dateFromString': {'dateString': '$Date'}}
        }
    },
    {
        '$match': {
            '$expr': {'$eq': [{'$month': '$DateParsed'}, 1]}
        }
    },
    {
        '$group': {
            '_id': '$City',
            'Avg_Temperature_F': {'$avg': '$Temperature_F'},
            'Avg_Humidity':      {'$avg': '$Humidity_%'}
        }
    },
    {
        '$project': {
            '_id': 0,
            'City':               '$_id',
            'Avg_Temperature_F':  {'$round': ['$Avg_Temperature_F', 2]},
            'Avg_Humidity':       {'$round': ['$Avg_Humidity', 2]}
        }
    }
]

results_q1 = list(weather_col.aggregate(pipeline_q1))
cities_q1, temps_q1, humidity_q1 = [], [], []
for doc in results_q1:
    cities_q1.append(doc['City'])
    temps_q1.append(doc['Avg_Temperature_F'])
    humidity_q1.append(doc['Avg_Humidity'])

print('Q1 data:')
for doc in results_q1:
    print(doc)

Q1 data:
{'City': 'Phoenix', 'Avg_Temperature_F': 71.71, 'Avg_Humidity': 31.39}
{'City': 'Seattle', 'Avg_Temperature_F': 55.35, 'Avg_Humidity': 78.64}
{'City': 'Chicago', 'Avg_Temperature_F': 50.29, 'Avg_Humidity': 63.65}
{'City': 'Denver', 'Avg_Temperature_F': 58.21, 'Avg_Humidity': 50.88}
{'City': 'Miami', 'Avg_Temperature_F': 78.81, 'Avg_Humidity': 84.32}


In [8]:
# Q2 Chicago monthly precipitation for 2025 (pie chart)
month_names = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

pipeline_q2 = [
    {
        '$addFields': {
            'DateParsed': {'$dateFromString': {'dateString': '$Date'}}
        }
    },
    {'$match': {'City': 'Chicago'}},
    {
        '$group': {
            '_id':            {'$month': '$DateParsed'},
            'Total_Precip':   {'$sum': '$Precipitation_in'}
        }
    },
    {'$sort': {'_id': 1}},
    {
        '$project': {
            '_id': 0,
            'Month':        '$_id',
            'Total_Precip': {'$round': ['$Total_Precip', 2]}
        }
    }
]

results_q2 = list(weather_col.aggregate(pipeline_q2))
months_q2, precip_q2 = [], []
for doc in results_q2:
    months_q2.append(month_names[doc['Month'] - 1])
    precip_q2.append(doc['Total_Precip'])

print('Q2 data:')
for doc in results_q2:
    print(doc)

Q2 data:
{'Month': 1, 'Total_Precip': 57.07}
{'Month': 2, 'Total_Precip': 56.75}
{'Month': 3, 'Total_Precip': 64.22}
{'Month': 4, 'Total_Precip': 13.71}
{'Month': 5, 'Total_Precip': 13.2}
{'Month': 6, 'Total_Precip': 13.13}
{'Month': 7, 'Total_Precip': 14.96}
{'Month': 8, 'Total_Precip': 15.55}
{'Month': 9, 'Total_Precip': 13.52}
{'Month': 10, 'Total_Precip': 13.38}
{'Month': 11, 'Total_Precip': 66.15}
{'Month': 12, 'Total_Precip': 75.91}


In [9]:
# Q3 – Monthly avg windspeed for cities with land area > 200 sq miles
# First get qualifying cities from cities collection
def clean_float(val):
    try:
        return float(re.sub(r'[^\d.]', '', str(val)))
    except:
        return 0.0

cities_data = list(cities_col.find({}, {'_id': 0}))
large_cities = [d['City'] for d in cities_data
                if clean_float(d.get('Land_area_mi2_2020', '0')) > 200]
print('Cities with land area > 200 sq mi:', large_cities)

pipeline_q3 = [
    {
        '$addFields': {
            'DateParsed': {'$dateFromString': {'dateString': '$Date'}}
        }
    },
    {'$match': {'City': {'$in': large_cities}}},
    {
        '$group': {
            '_id': {
                'City':  '$City',
                'Month': {'$month': '$DateParsed'}
            },
            'Avg_WindSpeed': {'$avg': '$WindSpeed_mph'}
        }
    },
    {'$sort': {'_id.Month': 1}},
    {
        '$project': {
            '_id': 0,
            'City':          '$_id.City',
            'Month':         '$_id.Month',
            'Avg_WindSpeed': {'$round': ['$Avg_WindSpeed', 2]}
        }
    }
]

results_q3 = list(weather_col.aggregate(pipeline_q3))
cities_q3, months_q3, wind_q3 = [], [], []
for doc in results_q3:
    cities_q3.append(doc['City'])
    months_q3.append(month_names[doc['Month'] - 1])
    wind_q3.append(doc['Avg_WindSpeed'])

df_q3 = pd.DataFrame({'City': cities_q3, 'Month': months_q3, 'Avg_WindSpeed': wind_q3})
print(df_q3.head(10))

Cities with land area > 200 sq mi: ['Chicago', 'Phoenix']
      City Month  Avg_WindSpeed
0  Phoenix   Jan           6.86
1  Chicago   Jan           6.70
2  Phoenix   Feb           7.59
3  Chicago   Feb           7.33
4  Chicago   Mar           6.57
5  Phoenix   Mar           6.54
6  Chicago   Apr           6.97
7  Phoenix   Apr           6.93
8  Phoenix   May           7.26
9  Chicago   May           6.85


In [10]:
# Q4 – Population 2020 vs 2025 for all 5 cities
def clean_int(val):
    try:
        return int(re.sub(r'[^\d]', '', str(val)))
    except:
        return 0

pipeline_q4 = [
    {'$project': {
        '_id': 0,
        'City':                     1,
        'Population_2025_estimate': 1,
        'Population_2020_census':   1
    }}
]

results_q4 = list(cities_col.aggregate(pipeline_q4))
cities_q4  = []
pop_2020   = []
pop_2025   = []
for doc in results_q4:
    cities_q4.append(doc['City'])
    pop_2020.append(clean_int(doc['Population_2020_census']))
    pop_2025.append(clean_int(doc['Population_2025_estimate']))

q4_df = pd.DataFrame({'City': cities_q4, 'Pop_2020': pop_2020, 'Pop_2025': pop_2025})
print(q4_df)
print('\n2025 Highest:', q4_df.loc[q4_df['Pop_2025'].idxmax(), 'City'])
print('2025 Lowest: ', q4_df.loc[q4_df['Pop_2025'].idxmin(), 'City'])
print('2020 Highest:', q4_df.loc[q4_df['Pop_2020'].idxmax(), 'City'])
print('2020 Lowest: ', q4_df.loc[q4_df['Pop_2020'].idxmin(), 'City'])

      City  Pop_2020  Pop_2025
0  Chicago   2746388   2731585
1  Phoenix   1608139   1665481
2  Seattle    737015    784777
3   Denver    715522    740613
4    Miami    442241    489812

2025 Highest: Chicago
2025 Lowest:  Miami
2020 Highest: Chicago
2020 Lowest:  Miami


In [11]:
# Q5 – Number of days with temperature > 70°F per city
pipeline_q5 = [
    {'$match': {'Temperature_F': {'$gt': 70}}},
    {
        '$group': {
            '_id':  '$City',
            'Days': {'$sum': 1}
        }
    },
    {'$sort': {'Days': -1}},
    {
        '$project': {
            '_id': 0,
            'City': '$_id',
            'Days': 1
        }
    }
]

results_q5 = list(weather_col.aggregate(pipeline_q5))
cities_q5, days_q5 = [], []
for doc in results_q5:
    cities_q5.append(doc['City'])
    days_q5.append(doc['Days'])

print('Q5 data:')
for doc in results_q5:
    print(doc)
print('\nCity with most days above 70°F:', cities_q5[0])

Q5 data:
{'Days': 223, 'City': 'Miami'}
{'Days': 180, 'City': 'Phoenix'}
{'Days': 44, 'City': 'Denver'}
{'Days': 18, 'City': 'Seattle'}

City with most days above 70°F: Miami


In [12]:
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1(children='Weather Analysis', style={'textAlign': 'center'}),

    html.Div([
        html.Label('Select Option:'),
        dcc.Dropdown(
            id='question-dropdown',
            options=[
                {'label': 'Avg Humidity and Temperature',            'value': 'q1'},
                {'label': 'Chicago Precipitation proportion',        'value': 'q2'},
                {'label': 'Monthly average windspeed - Landarea > 200 sq. miles', 'value': 'q3'},
                {'label': 'Population by 2020 and 2025',            'value': 'q4'},
                {'label': 'Peak wind speed',                        'value': 'q5'},
            ],
            value='q1',
            clearable=False,
            style={'width': '50%'}
        )
    ], style={'padding': '20px'}),

    dcc.Graph(id='main-chart')
])

In [13]:
@app.callback(
    Output('main-chart', 'figure'),
    Input('question-dropdown', 'value')
)
def update_chart(selected):

    if selected == 'q1':
        return {
            'data': [
                {'x': cities_q1, 'y': temps_q1,    'type': 'bar', 'name': 'Avg Temperature (F)'},
                {'x': cities_q1, 'y': humidity_q1, 'type': 'bar', 'name': 'Avg Humidity (%)'}
            ],
            'layout': {
                'title':   'Avg Temperature & Humidity by City (January 2024)',
                'xaxis':   {'title': 'City'},
                'yaxis':   {'title': 'Value'},
                'barmode': 'group'
            }
        }

    elif selected == 'q2':
        return {
            'data': [
                {'labels': months_q2, 'values': precip_q2, 'type': 'pie'}
            ],
            'layout': {
                'title': 'Chicago Monthly Precipitation Distribution (2024)'
            }
        }

    elif selected == 'q3':
        traces = []
        for city in q3_df['City'].unique():
            city_data = q3_df[q3_df['City'] == city]
            traces.append({
                'x':    city_data['Month'].tolist(),
                'y':    city_data['Avg_WindSpeed'].tolist(),
                'type': 'scatter',
                'mode': 'lines+markers',
                'name': city
            })
        return {
            'data': traces,
            'layout': {
                'title': 'Monthly Avg Windspeed - Land Area > 200 sq. miles',
                'xaxis': {'title': 'Month'},
                'yaxis': {'title': 'Avg Wind Speed (mph)'}
            }
        }

    elif selected == 'q4':
        max_2025 = q4_df.loc[q4_df['Pop_2025'].idxmax(), 'City']
        min_2025 = q4_df.loc[q4_df['Pop_2025'].idxmin(), 'City']
        max_2020 = q4_df.loc[q4_df['Pop_2020'].idxmax(), 'City']
        min_2020 = q4_df.loc[q4_df['Pop_2020'].idxmin(), 'City']
        return {
            'data': [
                {'x': q4_df['City'].tolist(), 'y': q4_df['Pop_2020'].tolist(),
                 'type': 'bar', 'name': '2020 Census'},
                {'x': q4_df['City'].tolist(), 'y': q4_df['Pop_2025'].tolist(),
                 'type': 'bar', 'name': '2025 Estimate'}
            ],
            'layout': {
                'title':   (f'Population by City: 2020 vs 2025 | '
                            f'2025 Highest: {max_2025}, Lowest: {min_2025} | '
                            f'2020 Highest: {max_2020}, Lowest: {min_2020}'),
                'xaxis':   {'title': 'City'},
                'yaxis':   {'title': 'Population'},
                'barmode': 'group'
            }
        }

    elif selected == 'q5':
        colors = ['crimson' if c == cities_q5[0] else 'steelblue' for c in cities_q5]
        return {
            'data': [
                {'x': cities_q5, 'y': days_q5, 'type': 'bar',
                 'name': 'Days > 70F',
                 'marker': {'color': colors}}
            ],
            'layout': {
                'title': f'Days with Temperature > 70F per City | Most: {cities_q5[0]}',
                'xaxis': {'title': 'City'},
                'yaxis': {'title': 'Number of Days'}
            }
        }

In [14]:
if __name__ == '__main__':
    app.run(debug=True, use_reloader=False)

Dash is running on http://127.0.0.1:8050/

 * Serving Flask app '__main__'
 * Debug mode: on


Address already in use
Port 8050 is in use by another program. Either identify and stop that program, or start the server with a different port.


SystemExit: 1

/opt/anaconda3/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
